# SparcNet Smoke Test (PyHealth 2.0)

This notebook uses the built-in `SleepEDFDataset` and `SleepStagingSleepEDF` task to load a small local `SleepEDF_demo` folder and run a forward-pass smoke test on the `SparcNet` model.

## Setup

The `SleepEDF_demo/` folder must have the following structure:
```
SleepEDF_demo/
  SC-subjects.xls              # from https://physionet.org/files/sleep-edfx/1.0.0/SC-subjects.xls
  sleepedf-cassette-pyhealth.csv  # auto-generated, then filtered to demo subjects only
  sleep-cassette/
    SC4001E0-PSG.edf
    SC4001EC-Hypnogram.edf
    SC4002E0-PSG.edf
    SC4002EC-Hypnogram.edf
    SC4011E0-PSG.edf
    SC4011EH-Hypnogram.edf
```


In [ ]:
import torch
from pathlib import Path
from pyhealth.datasets import SleepEDFDataset, get_dataloader
from pyhealth.models.sparcnet import SparcNet

## Load SleepEDF Dataset and Set Task

Load the SleepEDF cassette subset and apply the default `SleepStagingSleepEDF` task, which extracts 30-second EEG epochs labeled with sleep stages (W, N1, N2, N3, N4, R).

In [ ]:
sleepedf_root = str(Path("../SleepEDF_demo").resolve())

dataset = SleepEDFDataset(root=sleepedf_root, subset="cassette")
task_dataset = dataset.set_task()

print(f"Total samples: {len(task_dataset)}")
sample = task_dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Signal shape: {tuple(sample['signal'].shape)}")
print(f"Label classes: {sorted(set(int(task_dataset[i]['label']) for i in range(len(task_dataset))))}")

## Initialize Model and Forward Pass

Create a dataloader, initialize SparcNet from the dataset, and run a forward pass to verify the model produces loss and class probabilities.

In [ ]:
loader = get_dataloader(task_dataset, batch_size=8, shuffle=False)
batch = next(iter(loader))

print("batch keys:", list(batch.keys()))
print("batch['signal'] shape:", tuple(batch["signal"].shape))
print("batch['label'] shape:", tuple(batch["label"].shape))

model = SparcNet(dataset=task_dataset)

with torch.no_grad():
    ret = model(**batch)

print("loss:", float(ret["loss"]))
print("y_prob shape:", tuple(ret["y_prob"].shape))

## Create Train/Val/Test Data Loaders

Create data loaders for training, validation, and evaluation.

In [ ]:
train_loader = get_dataloader(task_dataset, batch_size=32, shuffle=True)
val_loader = get_dataloader(task_dataset, batch_size=8, shuffle=False)
test_loader = get_dataloader(task_dataset, batch_size=8, shuffle=False)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

## Train and Evaluate

Train SparcNet for 1 epoch and evaluate on the test set.

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model, enable_logging=True)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=1,
)

metrics = trainer.evaluate(test_loader)
print("Trainer eval metrics:", metrics)